[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/mechanical_work/mechanical_work.ipynb)

# Mechanical Work in a Mirror

This example computes the power a push delivers to a turning crank, and the power the mirrored push delivers to the mirrored crank. Mechanical work measures how an applied force drives a motion, and power is the rate at which it does so. The mirrored crank turns the other way, yet the mirrored push supplies the same positive power.

In projective geometric algebra the handle is modelled as a point and the push's direction as an ideal point, both intrinsic antivectors, and the crank's rate of turning as an extrinsic bivector. Joining the handle to the push gives the forque on the crank, `handle & force`, the intrinsic line the force acts along, and the power is its pairing with the rate, `forque & rate`. The handle's velocity is its commutator with the rate, `handle.commutator(rate)`. A mirror acts on every ingredient alike, `mirror >> rate`, and because an intrinsic forque pairs with an extrinsic rate, the power is an extrinsic scalar that the reflection leaves unchanged.

In the vector notation of mechanics, the torque about the pivot is $\boldsymbol\tau = \mathbf r \times \mathbf F$, the handle's velocity is $\mathbf v = \boldsymbol\omega \times \mathbf r$, and the power is $P = \boldsymbol\tau \cdot \boldsymbol\omega = \mathbf F \cdot \mathbf v$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import NumpyContext, stack
from numga.algebras import PGA2D
from examples.util.animation import show_animation
from examples.orientation.mechanical_work import render

In [ ]:
ga = PGA2D
mv = NumpyContext(ga).multivector
# Trailing type comments mark + extrinsic and - intrinsic orientation.

## 1. Power delivered to a crank and its mirror image

A push along the handle's path follows each handle as the crank turns, and its forque paired with the rate, `forque & rate`, is the power it delivers. The mirrored crank is the reflection of the handle, the push and the rate, and the same lines give its power.

In [ ]:
separation = 1.65
radius = 0.9
speed = 1
force_strength = 1.5
n_steps = 36
duration_ms = 85
times = np.linspace(0, 2 * np.pi / speed, n_steps, endpoint=False)

In [ ]:
mirror = mv.x                                               # [] Plane+
pivot = mv.point([-separation, 0])                          # [] Point-
handle = mv.point([-separation + radius, 0])                # [] Point-
force = mv.direction([0, force_strength])                   # [] Direction-

# These two planes meet at the pivot and generate its rotation.
rate = ((mv.x + separation * mv.w) ^ mv.y) * speed          # [] Rate+

# Reflect the physical ingredients before computing either crank's power.
pivots = stack([pivot, mirror >> pivot])                    # [cranks] Point-
handles = stack([handle, mirror >> handle])                 # [cranks] Point-
forces = stack([force, mirror >> force])                    # [cranks] Direction-
rates = stack([rate, mirror >> rate])                       # [cranks] Rate+

poses = (rates[None, :] * (-times[:, None] / 2)).exp()      # [steps, cranks] Motor+
handles = poses >> handles                                  # [steps, cranks] Point-
forces = poses >> forces                                    # [steps, cranks] Direction-

# The force acts at the moving handle, whose velocity follows from the rate.
velocities = handles.commutator(rates)                      # [steps, cranks] Direction-
forques = handles & forces                                  # [steps, cranks] Forque-
powers = forques & rates                                    # [steps, cranks] Scalar+

print("Crank power", powers[0, 0].coefficient_table(), sep="\n")
print("Mirrored crank power", powers[0, 1].coefficient_table(), sep="\n")
show_animation(render.animate(pivots, handles, velocities, forces), "mechanical_work", duration_ms)

Teal arrows show each handle's velocity, and orange arrows show the applied force; the dashed line marks the mirror plane. Both pushes act along the motion and supply the same positive power throughout the turn. The forque is an intrinsic line and the rate an extrinsic bivector, so their pairing, `forque & rate`, is an extrinsic scalar, which the reflection leaves as it is.